# Train: Logistic Regression on the 200k Dataset (cmu + synthetic)

## 1. Introduction

This is step 3 (**Train**) of the five-step workflow, for the model trained on the 200k set from `01_prepare_synthetic`. The test set is **not opened** in this notebook. We repeat the steps of the 70k model so that the two models can be compared fairly:

1. Load the training set and define X and y.
2. Train **baseline models**.
3. Choose the features, using training data only.
4. Train the logistic regression and report its **training score**.
5. Estimate how well it will do on new people with **5-fold cross-validation**.
6. Choose the **decision threshold** for our Recall goal.
7. Save the model.

### The problem that synthetic data creates for cross-validation
The 70k notebook used ordinary cross-validation. That would be **wrong here**. A synthetic row is an interpolation between real people. If a synthetic row sits in the training part of a fold and the real person it was built from sits in the validation part, the model has effectively seen that person, and the score is too good.

The fix has three parts:

| Rule | Why |
|---|---|
| **Validate on real rows only** | Synthetic rows are never used to score a model |
| **Split the real rows into folds first** | The real people are divided before any synthetic row exists |
| **Create the synthetic rows inside each fold, from the training part only** | No synthetic row can be built from a validation person |

So in every fold the model is trained on real rows + synthetic rows made from those real rows, and scored on real people it has never seen, in the same way as the final model will be scored on the test set.

### Metrics on a 50/50 scale
The real validation rows are not 50/50 (the real no-diabetes rows outnumber the diabetes rows about 3.6 to 1). **Recall, specificity and ROC AUC do not depend on the mix**, and from them we compute precision, F1 and accuracy *as if* the validation data were 50/50. This puts every score on the same scale as the 70k notebooks and the balanced test set.

**Source table:** `diabetes_cmu_synth_train` | **Output:** a saved model file

## 2. Import Libraries and Connect

> **Model refresh note:** This notebook now trains with the shared 14-feature list and excludes `CholCheck` and `AnyHealthcare`. Previously saved result text and outputs were produced with an earlier feature set; rerun this notebook after loading the saved CSVs to refresh them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
import sklearn

# statsmodels: p-values for the feature selection
import statsmodels.api as sm

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sqlalchemy import create_engine, text

# Our own helper file in this folder (creates the synthetic rows)
from synth_utils import generate_synthetic

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

engine = create_engine("mysql+pymysql://root:@localhost:3306/diabetes_final_db")

TRAIN_TABLE = "diabetes_cmu_synth_train"
TARGET = "Diabetes_binary"
RANDOM_STATE = 42
TARGET_RECALL = 0.85              # PLACEHOLDER: same working goal as the 70k model (the group should confirm it)
REAL_PREVALENCE = 0.14            # about 14% of the surveyed people had diabetes
MODEL_PATH = Path("../../application/models/logreg_200k.joblib")

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("Libraries imported and database connection successful.")

## 3. Load the Training Set

In [ ]:
train_df = pd.read_sql(f"SELECT * FROM {TRAIN_TABLE}", con=engine)

all_features = [c for c in train_df.columns if c not in (TARGET, "is_synthetic")]
X_train, y_train = train_df[all_features], train_df[TARGET]

# The real rows only (used for cross-validation and for the p-values)
real_df = train_df[train_df["is_synthetic"] == 0].reset_index(drop=True)
X_real, y_real = real_df[all_features], real_df[TARGET]

NON_BINARY = ["BMI", "GenHlth", "MentHlth", "PhysHlth", "Age", "Education", "Income"]

summary = pd.DataFrame({
    "Item": ["Training rows", "Features", "No diabetes (0)", "Diabetes (1)",
             "Real rows", "Real: no diabetes", "Real: diabetes", "Synthetic rows (all diabetes)"],
    "Value": [f"{len(train_df):,}", len(all_features), f"{(y_train == 0).sum():,}", f"{(y_train == 1).sum():,}",
              f"{len(real_df):,}", f"{(y_real == 0).sum():,}", f"{(y_real == 1).sum():,}",
              f"{int(train_df['is_synthetic'].sum()):,}"],
})
display(summary.style.hide(axis="index"))

print(f"Training set '{TRAIN_TABLE}' loaded successfully.")

## 4. Tools Used in This Notebook

- **Cross-validation folds (real rows only, synthetic rows made inside each fold).** The folds are built **once** and reused for every model, so all models are scored on exactly the same people.
- **Model builder:** the same as for the 70k model: logistic regression with no penalty, the 7 non-binary features standardized inside the model.
- **Metrics on a 50/50 scale:** Recall, specificity and ROC AUC are measured directly. Precision, F1 and accuracy are computed from Recall and the false alarm rate as if the people with and without diabetes were equally many.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def make_logreg(cols):
    """Logistic regression (no penalty) with scaling of the non-binary columns done inside the model."""
    scale = [c for c in cols if c in NON_BINARY]
    prep = ColumnTransformer([("scale", StandardScaler(), scale)],
                             remainder="passthrough", verbose_feature_names_out=False)
    return Pipeline([("prep", prep), ("model", LogisticRegression(C=np.inf, max_iter=2000))])

def scores_5050(y, proba, t=0.5):
    """Metrics for one set of rows, with precision/F1/accuracy on a 50/50 scale."""
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    tpr, tnr = tp / (tp + fn), tn / (tn + fp)
    prec = tpr / (tpr + (1 - tnr)) if (tpr + (1 - tnr)) > 0 else 0.0
    f1 = 2 * prec * tpr / (prec + tpr) if (prec + tpr) > 0 else 0.0
    return {"Accuracy": (tpr + tnr) / 2, "Recall": tpr, "Specificity": tnr,
            "Precision": prec, "F1": f1, "ROC AUC": roc_auc_score(y, proba)}

# --- Build the 5 folds ONCE: split the real rows, then create synthetic rows from each training part ---
folds = []
for i, (tr, va) in enumerate(cv.split(X_real, y_real)):
    X_tr, y_tr = X_real.iloc[tr], y_real.iloc[tr]
    X_va, y_va = X_real.iloc[va], y_real.iloc[va]
    n_new = int((y_tr == 0).sum() - (y_tr == 1).sum())                 # make the training part 50/50
    syn = generate_synthetic(X_tr[y_tr == 1], n_new, seed=RANDOM_STATE + i, avoid=X_va)
    folds.append({
        "X_tr": pd.concat([X_tr, syn], ignore_index=True),
        "y_tr": pd.concat([y_tr, pd.Series(1, index=range(n_new))], ignore_index=True),
        "X_va": X_va, "y_va": y_va,
    })

def cv_scores(build_model, cols):
    """Mean and standard deviation of every metric over the 5 folds (validation rows are real)."""
    rows = []
    for f in folds:
        model = build_model(cols).fit(f["X_tr"][cols], f["y_tr"])
        rows.append(scores_5050(f["y_va"], model.predict_proba(f["X_va"][cols])[:, 1]))
    res = pd.DataFrame(rows)
    return {m: (res[m].mean(), res[m].std()) for m in res.columns}

fold_info = pd.DataFrame({
    "Fold": range(1, 6),
    "Training rows (real + synthetic)": [f"{len(f['X_tr']):,}" for f in folds],
    "of which synthetic": [f"{len(f['X_tr']) - (len(X_real) - len(f['X_va'])):,}" for f in folds],
    "Validation rows (all real)": [f"{len(f['X_va']):,}" for f in folds],
    "Validation: % diabetes": [f"{f['y_va'].mean() * 100:.1f}%" for f in folds],
})
display(fold_info.style.hide(axis="index"))

print("Tools defined and the 5 cross-validation folds built.")

## 5. Baseline Models

The same three baselines as for the 70k model. They are trained on each fold's training part and scored on the real validation people. All three use no information from the features, so every one should sit at chance.

In [ ]:
baselines = {
    "Baseline: always no diabetes": lambda cols: DummyClassifier(strategy="constant", constant=0),
    "Baseline: flag everybody": lambda cols: DummyClassifier(strategy="constant", constant=1),
    "Baseline: random guess": lambda cols: DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
}

baseline_cv = {name: cv_scores(b, all_features) for name, b in baselines.items()}

table = pd.DataFrame({
    name: {m: f"{mean:.3f} ± {sd:.3f}" for m, (mean, sd) in scores.items()}
    for name, scores in baseline_cv.items()
}).T
display(table)

print("Baselines trained successfully.")

**Interpretation:** The baselines set reference points for the synthetic-data experiment. An always-positive rule can achieve perfect recall while flagging everyone, so improvements must be considered together with specificity and precision. Cross-validation validation rows are real respondents; synthetic rows are used only inside each training fold.

In [ ]:
# The app and both saved models use this identical feature list.
# CholCheck and AnyHealthcare are excluded because they capture access to/detection of care,
# which can distort a screening score. This set also preserves direct comparability.
FEATURES = ['HighBP', 'HighChol', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'Fruits', 'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Income']
chosen_name = "Shared 14-feature set (healthcare-access proxies excluded)"
missing_features = [f for f in FEATURES if f not in all_features]
if missing_features:
    raise ValueError(f"Training table is missing shared features: {missing_features}")
print(f"Chosen feature set: {chosen_name} ({len(FEATURES)} features)")
print("Features:", ", ".join(FEATURES))

**Interpretation:** M2 uses the same 14 predictors as M1, so the later comparison changes the training data rather than the feature list. `CholCheck` and `AnyHealthcare` are excluded as healthcare-access/detection proxies; the fixed list is not re-selected using the test set.

In [ ]:
logreg = make_logreg(FEATURES)
logreg.fit(X_train[FEATURES], y_train)

proba_all = logreg.predict_proba(X_train[FEATURES])[:, 1]
is_real = (train_df["is_synthetic"] == 0).to_numpy()

logreg_train = {"All 200,000 rows": scores_5050(y_train, proba_all),
                "Real rows only": scores_5050(y_train[is_real], proba_all[is_real])}
display(pd.DataFrame(logreg_train).round(3))

# Odds ratios of the final model
names = logreg.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(logreg.named_steps["model"].coef_[0], index=names)
odds = pd.DataFrame({"Odds ratio": np.exp(coefs)}).sort_values("Odds ratio", ascending=False).round(3)
display(odds)

print("Logistic regression trained successfully.")

**Interpretation:** The 200,000-row fit is scored both on all training rows and on its real subset, but these are training scores and are optimistic. Synthetic-row performance is not an independent validation; use the five-fold results on real validation respondents and the shared held-out test set for evidence of generalization.

In [ ]:
logreg_cv = cv_scores(make_logreg, FEATURES)

all_cv = {**baseline_cv, "Logistic regression": logreg_cv}
cv_table = pd.DataFrame({
    name: {m: f"{mean:.3f} ± {sd:.3f}" for m, (mean, sd) in scores.items()}
    for name, scores in all_cv.items()
}).T
print("Cross-validation scores (real validation people, 50/50 scale):")
display(cv_table)

gap = pd.DataFrame({
    "Training score (real rows)": pd.Series(logreg_train["Real rows only"]),
    "Cross-validation score": pd.Series({m: v[0] for m, v in logreg_cv.items()}),
})
gap["Gap (training - CV)"] = gap["Training score (real rows)"] - gap["Cross-validation score"]
print("Logistic regression: training vs cross-validation")
display(gap.round(4))

beats = all(logreg_cv["ROC AUC"][0] > b["ROC AUC"][0] for b in baseline_cv.values())
print("Logistic regression beats every baseline on ROC AUC:", beats)

**Interpretation:** This cross-validation estimates performance on real validation respondents while synthetic examples are generated only from the corresponding training fold. Compare the logistic model with the baselines here; the held-out test set is still needed for the final model comparison.

In [ ]:
# Out-of-fold probabilities for every real training person
oof_p, oof_y = [], []
for f in folds:
    model = make_logreg(FEATURES).fit(f["X_tr"][FEATURES], f["y_tr"])
    oof_p.append(model.predict_proba(f["X_va"][FEATURES])[:, 1])
    oof_y.append(f["y_va"].to_numpy())
oof_p, oof_y = np.concatenate(oof_p), np.concatenate(oof_y)

def threshold_for(goal):
    """Highest threshold at which at least `goal` of the people with diabetes are flagged."""
    return float(np.quantile(oof_p[oof_y == 1], 1 - goal, method="lower"))

def metrics_at(t):
    flagged = oof_p >= t
    tpr, fpr = flagged[oof_y == 1].mean(), flagged[oof_y == 0].mean()
    flagged_real = tpr * REAL_PREVALENCE + fpr * (1 - REAL_PREVALENCE)
    return {"Recall": tpr, "Specificity": 1 - fpr,
            "Precision (50/50 scale)": tpr / (tpr + fpr),
            f"Precision at {REAL_PREVALENCE:.0%} prevalence": tpr * REAL_PREVALENCE / flagged_real,
            f"Share of people flagged at {REAL_PREVALENCE:.0%}": flagged_real}

rows = [{"Setting": "Default threshold", "Threshold": 0.5, **metrics_at(0.5)}]
for goal in [0.70, 0.80, 0.85, 0.90]:
    t = threshold_for(goal)
    rows.append({"Setting": f"Recall goal {goal:.0%}", "Threshold": t, **metrics_at(t)})
goals = pd.DataFrame(rows)
display(goals.round(3).style.hide(axis="index"))

chosen_threshold = threshold_for(TARGET_RECALL)

# Chart: recall, specificity and precision as the threshold changes
grid = np.linspace(0.05, 0.95, 91)
curve = pd.DataFrame([metrics_at(t) for t in grid], index=grid)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(grid, curve["Recall"], label="Recall", color="#E45756")
ax.plot(grid, curve["Specificity"], label="Specificity", color="#54A24B")
ax.plot(grid, curve["Precision (50/50 scale)"], label="Precision (50/50 scale)", color="#4C78A8")
ax.axhline(TARGET_RECALL, ls=":", color="grey", label=f"Recall goal ({TARGET_RECALL:.0%})")
ax.axvline(chosen_threshold, ls="--", color="black", label=f"Chosen threshold ({chosen_threshold:.2f})")
ax.axvline(0.5, ls="--", color="lightgrey", label="Default threshold (0.50)")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score (real training people, out-of-fold)")
ax.set_title("Recall, specificity and precision as the threshold changes")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Chosen threshold: {chosen_threshold:.3f} (reaches a Recall of at least {TARGET_RECALL:.0%} out-of-fold).")

**Interpretation:** The chosen threshold is about 0.3755, derived from out-of-fold predictions for real training respondents to target 85% recall. It is fixed before the test table is opened; the 200k model’s test recall is about 84.9%, with specificity about 59.9%.

In [ ]:
bundle = {
    "model": logreg,
    "features": FEATURES,
    "threshold": chosen_threshold,
    "target_recall": TARGET_RECALL,
    "feature_set_name": chosen_name,
    "random_state": RANDOM_STATE,
    "sklearn_version": sklearn.__version__,
    "training_data": "200k: 100,000 no diabetes (real) + 100,000 diabetes (real + synthetic)",
}
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, MODEL_PATH)

# Verify: reload the file and check that it gives identical predictions
reloaded = joblib.load(MODEL_PATH)
same = np.allclose(reloaded["model"].predict_proba(X_train[FEATURES])[:, 1],
                   logreg.predict_proba(X_train[FEATURES])[:, 1])

saved = pd.DataFrame({
    "Item": ["File", "Features saved", "Threshold saved", "scikit-learn version", "Reloaded model gives identical predictions"],
    "Value": [str(MODEL_PATH), len(reloaded["features"]), round(reloaded["threshold"], 3),
              reloaded["sklearn_version"], same],
})
display(saved.style.hide(axis="index"))

print("Model saved successfully.")

## 11. Summary

This notebook trains the 200k real-plus-synthetic logistic regression model using the same 14-feature list as the 70k model. Synthetic rows are created inside each real-only cross-validation fold for threshold selection. The final model is trained on the saved real-plus-synthetic training table and saved to `application/models/logreg_200k.joblib`. Rerun the notebook to refresh all metrics and charts after the feature-set change.